In [1]:
# This notebook answers RQ3: Which cantons offer the best opportunities for combining different outdoor activities (hiking, cycling, mountain biking, skating)?

import pandas as pd
# pandas: works with tables (DataFrames).
# Used to load the routes, count them per canton and activity, and calculate percentile ranks.

In [2]:
# The four route tables with cantons (result of canton_assignment.ipynb) are stored in a list.
# The four route tables with cantons (result of canton_assignment.ipynb)
files = [
    "result_wanderland_cantons.csv",
    "result_veloland_cantons.csv",
    "result_mountainbikeland_cantons.csv",
    "result_skatingland_cantons.csv",
]

In [3]:
routes = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

In [4]:
# Check: number of routes per activity and in total
print(routes["land"].value_counts())
print("Total:", len(routes))

land
mountainbikeland    286
wanderland          274
veloland             65
skatingland          11
Name: count, dtype: int64
Total: 636


In [5]:
# One row per route and canton (a route counts for every canton in which it starts or ends),
# together with the activity type of the route.
route_canton = pd.concat([
    routes[["start_canton", "land"]].rename(columns={"start_canton": "canton"}),
    routes[["end_canton", "land"]].rename(columns={"end_canton": "canton"}),
])

# A route that starts and ends in the same canton counts only once for that canton
route_canton = route_canton.reset_index().drop_duplicates()

# Routes abroad are not counted
route_canton = route_canton[route_canton["canton"] != "abroad"]

# Check: number of route-canton combinations
print("Route-canton combinations:", len(route_canton))

Route-canton combinations: 648


In [6]:
# Number of routes per canton (rows) and activity (columns)
routes_per_activity = pd.crosstab(route_canton["canton"], route_canton["land"])

print(routes_per_activity.to_string())

land    mountainbikeland  skatingland  veloland  wanderland
canton                                                     
AG                     1            0         1           0
AI                     1            0         1           2
AR                     0            0         1           5
BE                    19            1        15          35
BL                     1            0         5           4
BS                     0            1         1           0
FR                     0            2         1          15
GE                     0            0         3           6
GL                     5            0         0          11
GR                    97            1         3          50
JU                    10            0         1           7
LU                     1            0         2           2
NE                     9            1         3           4
NW                     6            0         0          10
OW                     5            0   

In [7]:
# Load the geographic key figures of the cantons (BFS, as of 01.01.2026)
canton_figures = pd.read_excel("Geographische_Kennzahlen.xlsx", sheet_name="Daten")

# Canton abbreviation for each official canton number (order of the Federal Constitution)
canton_abbreviations = {
    1: "ZH", 2: "BE", 3: "LU", 4: "UR", 5: "SZ", 6: "OW", 7: "NW", 8: "GL", 9: "ZG",
    10: "FR", 11: "SO", 12: "BS", 13: "BL", 14: "SH", 15: "AR", 16: "AI", 17: "SG",
    18: "GR", 19: "AG", 20: "TG", 21: "TI", 22: "VD", 23: "VS", 24: "NE", 25: "GE", 26: "JU"
}
canton_figures["canton"] = canton_figures["KTNR"].map(canton_abbreviations)

# Convert the area from hectares to square kilometres (1 km² = 100 ha)
canton_figures["area_km2"] = canton_figures["AREA_HA"] / 100

# Check: number of cantons
print("Cantons:", len(canton_figures))

Cantons: 26


In [9]:
# Include all 26 cantons: cantons without any route (Zug) get 0 for every activity
routes_per_activity = routes_per_activity.reindex(canton_figures["canton"], fill_value=0)

# Area of each canton, with the canton abbreviation as row name
area = canton_figures.set_index("canton")["area_km2"]

# Density: number of routes per 100 km², calculated separately for each activity
density = routes_per_activity.div(area, axis=0) * 100

print(density.round(2).to_string())

land    mountainbikeland  skatingland  veloland  wanderland
canton                                                     
ZH                  0.00         0.12      0.12        0.29
BE                  0.32         0.02      0.25        0.59
LU                  0.07         0.00      0.13        0.13
UR                  1.76         0.00      0.00        0.56
SZ                  1.10         0.00      0.00        0.99
OW                  1.02         0.00      0.61        1.22
NW                  2.18         0.00      0.00        3.63
GL                  0.73         0.00      0.00        1.61
ZG                  0.00         0.00      0.00        0.00
FR                  0.00         0.12      0.06        0.90
SO                  0.25         0.25      0.38        0.76
BS                  0.00         2.71      2.71        0.00
BL                  0.19         0.00      0.97        0.77
SH                  0.00         0.34      1.68        1.68
AR                  0.00         0.00   

In [10]:
# Overview of the densities per activity, to decide on the category limits
print(density.describe().round(2).to_string())

# Number of cantons without any route per activity
print()
print("Cantons with density 0:")
print((density == 0).sum().to_string())

land   mountainbikeland  skatingland  veloland  wanderland
count             26.00        26.00     26.00       26.00
mean               0.57         0.15      0.40        0.91
std                0.62         0.53      0.62        0.80
min                0.00         0.00      0.00        0.00
25%                0.00         0.00      0.05        0.51
50%                0.38         0.00      0.14        0.73
75%                0.99         0.10      0.40        1.12
max                2.18         2.71      2.71        3.63

Cantons with density 0:
land
mountainbikeland     8
skatingland         16
veloland             5
wanderland           3


In [11]:
# Swiss average density per activity: all routes of an activity divided by the area of Switzerland.
# It serves as the reference value for the density categories of each canton.
swiss_area = area.sum()
swiss_density = routes_per_activity.sum() / swiss_area * 100

print("Area of Switzerland (km²):", round(swiss_area))
print(swiss_density.round(2).to_string())

Area of Switzerland (km²): 41291
land
mountainbikeland    0.69
skatingland         0.03
veloland            0.17
wanderland          0.68


In [12]:
# Ratio of each canton's density to the Swiss average of the same activity
# (1 = exactly the Swiss average, 2 = twice as dense as the Swiss average)
ratio = density.div(swiss_density, axis=1)

# Assign a category based on the ratio
def density_category(r):
    if r == 0:
        return "no offer"
    elif r < 0.75:
        return "below average"
    elif r <= 1.25:
        return "average"
    elif r <= 2:
        return "above average"
    else:
        return "high"

density_categories = ratio.map(density_category)

print(density_categories.to_string())

land   mountainbikeland    skatingland       veloland     wanderland
canton                                                              
ZH             no offer           high  below average  below average
BE        below average  below average  above average        average
LU        below average       no offer        average  below average
UR                 high       no offer       no offer        average
SZ        above average       no offer       no offer  above average
OW        above average       no offer           high  above average
NW                 high       no offer       no offer           high
GL              average       no offer       no offer           high
ZG             no offer       no offer       no offer       no offer
FR             no offer           high  below average  above average
SO        below average           high           high        average
BS             no offer           high           high       no offer
BL        below average       no o

In [13]:
# Combine routes, density, ratio and category into one table:
# one row per canton and activity (long format, practical for visualisations)
rq3_density = pd.DataFrame({
    "routes": routes_per_activity.stack(),
    "density_per_100km2": density.stack().round(2),
    "ratio_to_swiss_average": ratio.stack().round(2),
    "category": density_categories.stack(),
}).reset_index()

# Save the result
rq3_density.to_csv("rq3_result_density_categories.csv", index=False)

# Check: number of rows (26 cantons x 4 activities = 104) and first rows
print("Rows:", len(rq3_density))
print(rq3_density.head(8).to_string())

Rows: 104
  canton              land  routes  density_per_100km2  ratio_to_swiss_average       category
0     ZH  mountainbikeland       0                0.00                    0.00       no offer
1     ZH       skatingland       2                0.12                    3.41           high
2     ZH          veloland       2                0.12                    0.69  below average
3     ZH        wanderland       5                0.29                    0.43  below average
4     BE  mountainbikeland      19                0.32                    0.46  below average
5     BE       skatingland       1                0.02                    0.50  below average
6     BE          veloland      15                0.25                    1.51  above average
7     BE        wanderland      35                0.59                    0.87        average


In [15]:
# Share of each canton in the Swiss offer of each activity (in percent).
# Example: if 10 of all 286 mountain bike routes are in a canton, its share is 3.5 %.
# This makes the activities comparable, although they differ strongly in size
# (e.g. 286 mountain bike routes vs. 11 skating routes in Switzerland).
shares = routes_per_activity / routes_per_activity.sum() * 100

print(shares.round(1).to_string())

land    mountainbikeland  skatingland  veloland  wanderland
canton                                                     
ZH                   0.0         14.3       2.9         1.8
BE                   6.6          7.1      21.7        12.5
LU                   0.3          0.0       2.9         0.7
UR                   6.6          0.0       0.0         2.2
SZ                   3.5          0.0       0.0         3.2
OW                   1.7          0.0       4.3         2.2
NW                   2.1          0.0       0.0         3.6
GL                   1.7          0.0       0.0         3.9
ZG                   0.0          0.0       0.0         0.0
FR                   0.0         14.3       1.4         5.4
SO                   0.7         14.3       4.3         2.2
BS                   0.0          7.1       1.4         0.0
BL                   0.3          0.0       7.2         1.4
SH                   0.0          7.1       7.2         1.8
AR                   0.0          0.0   

In [16]:
import numpy as np
# numpy: library for numerical calculations; used here for the natural logarithm (np.log).

# 1. Profile of each canton: the four shares rescaled so that they add up to 1 (100 %) per canton.
#    This removes the size of the canton; only the distribution between the activities remains.
profile = shares.div(shares.sum(axis=1), axis=0)

# 2. Shannon index: measures how evenly the routes are distributed between the activities.
#    Activities without any route (share 0) are left out, as the logarithm of 0 is not defined.
shannon = -(profile * np.log(profile.where(profile > 0))).sum(axis=1, min_count=1)

# 3. Evenness: Shannon index divided by its maximum (log of the number of activities, here 4).
#    0 = all routes in one activity, 1 = all four activities equally represented.
evenness = (shannon / np.log(len(shares.columns))).round(2)

# Cantons without any route (Zug) get no value, since a distribution cannot be calculated
print(evenness.sort_values(ascending=False).to_string())

canton
BE    0.91
SG    0.91
NE    0.90
GR    0.81
JU    0.75
VD    0.74
OW    0.73
VS    0.71
SH    0.70
AI    0.69
TG    0.68
SO    0.68
TI    0.66
FR    0.57
LU    0.54
ZH    0.52
SZ    0.50
AR    0.50
NW    0.48
GE    0.46
GL    0.44
BL    0.43
UR    0.40
AG    0.36
BS    0.33
ZG     NaN


In [23]:
# Assign a diversity category based on the evenness.
# Cantons with fewer than 5 routes in total get no diversity category,
# since the distribution of so few routes is largely a matter of chance.
# Values exactly on a limit belong to the higher category.
MIN_ROUTES = 5
routes_total = routes_per_activity.sum(axis=1)

def diversity_category(canton):
    e = evenness[canton]
    if routes_total[canton] == 0:
        return "no offer"
    elif routes_total[canton] < MIN_ROUTES:
        return "too few routes"
    elif e < 0.5:
        return "low"
    elif e < 0.75:
        return "medium"
    else:
        return "high"

diversity_categories = pd.Series(
    {canton: diversity_category(canton) for canton in evenness.index}
)

print(diversity_categories.value_counts().to_string())

medium            12
high               5
low                5
too few routes     3
no offer           1


In [28]:
# Combine evenness, diversity category and total number of routes into one table
rq3_diversity = pd.DataFrame({
    "evenness": evenness,
    "category": diversity_categories,
    "routes_total": routes_per_activity.sum(axis=1),
}).sort_values("evenness", ascending=False).reset_index()

# Save the result
rq3_diversity.to_csv("rq3_result_diversity.csv", index=False)

# Check: number of cantons
print("Saved:", len(rq3_diversity), "cantons")

Saved: 26 cantons


In [27]:
print(rq3_diversity)

   index  evenness        category  routes_total
0     BE      0.91            high            70
1     SG      0.91            high            36
2     NE      0.90            high            17
3     GR      0.81            high           151
4     JU      0.75            high            18
5     VD      0.74          medium            48
6     OW      0.73          medium            14
7     VS      0.71          medium            79
8     SH      0.70          medium            11
9     AI      0.69  too few routes             4
10    TG      0.68          medium             8
11    SO      0.68          medium            13
12    TI      0.66          medium            42
13    FR      0.57          medium            18
14    LU      0.54          medium             5
15    ZH      0.52          medium             9
16    SZ      0.50          medium            19
17    AR      0.50          medium             6
18    NW      0.48             low            16
19    GE      0.46  

In [29]:
import pandas as pd
import canton_assignment

test_routes = pd.read_csv("old_csv/result_wanderland.csv")
cantons = canton_assignment.load_cantons()

result = canton_assignment.assign_cantons(test_routes, "Geodaten/Wanderland/Route.shp", cantons)

Scraped routes: 274
coordinates: 359
['land', 'url', 'title', 'start', 'via', 'end', 'distance_km', 'el_gain_m', 'el_loss_m', 'el_max_m', 'el_min_m', 'duration_min', 'direction_note', 'asphalt_km', 'natural_km', 'singletrail_km', 'technical', 'condition', 'facts_raw', 'details_raw']
Entries without a valid route number: 4
Route geometries after removing duplicates: 351
Scraped routes: 274
With matching geometry: 274
Duplicate route numbers: 0
Rows after linking: 274
geometry
LineString    274
Name: count, dtype: int64
Total: 274
Abroad: 6
start_canton
GR        50
VS        32
BE        32
VD        25
TI        16
FR        14
SG        13
GL        11
SZ         9
JU         7
GE         6
SO         6
NW         6
UR         6
abroad     6
OW         5
SH         5
ZH         5
AR         5
NE         4
BL         4
TG         4
LU         2
AI         1
Total: 274
Abroad: 7
end_canton
GR        50
BE        35
VS        32
VD        24
TI        16
FR        14
SG        13
GL     

In [30]:
import pandas as pd
import rq3
from utils import load_canton_figures

# Test only: the four route tables with cantons replace the pipeline for now
test_routes = pd.concat([
    pd.read_csv("old_csv/result_wanderland_cantons.csv"),
    pd.read_csv("old_csv/result_veloland_cantons.csv"),
    pd.read_csv("old_csv/result_mountainbikeland_cantons.csv"),
    pd.read_csv("old_csv/result_skatingland_cantons.csv"),
], ignore_index=True)

canton_figures = load_canton_figures()
routes_per_activity = rq3.count_routes_per_activity(test_routes, canton_figures)

density_table = rq3.density_categories(routes_per_activity, canton_figures)
diversity_table = rq3.diversity(routes_per_activity)

print("Density rows:", len(density_table))
print(diversity_table["category"].value_counts().to_string())

land    mountainbikeland  skatingland  veloland  wanderland
canton                                                     
AG                     1            0         1           0
AI                     1            0         1           2
AR                     0            0         1           5
BE                    19            1        15          35
BL                     1            0         5           4
BS                     0            1         1           0
FR                     0            2         1          15
GE                     0            0         3           6
GL                     5            0         0          11
GR                    97            1         3          50
JU                    10            0         1           7
LU                     1            0         2           2
NE                     9            1         3           4
NW                     6            0         0          10
OW                     5            0   